# Resume Clinic — E2E live-agent validation

ADR-066 Phase 6 companion. Walks the full out-of-graph clinic flow against the real backend:

1. Resolve the active profile and pick a resume.
2. Optionally set a target role / track.
3. `POST /users/{id}/resume-clinic` → inspect quality / alignment / overhaul JSON.
4. Show the fidelity verdict (the agent-policing layer).
5. Exercise an `approve` decision (and optionally an `edit` with a hand-crafted draft).
6. List past clinic runs.

## Preflight

- The FastAPI backend must be running locally (`uvicorn app.api.main:app --reload`). This notebook talks to it via HTTP.
- `ANTHROPIC_API_KEY` must be set in the backend's environment so it picks the real-agent dependency graph. Otherwise the clinic returns mocked output (which makes the structural checks pass but the semantic-drift inspection meaningless).
- The acting profile is whichever you set with `set_user_id` below; default is the system profile `"0"`.

## What this notebook is NOT

It is not a unit test. The mocked Phase-6 unit tests (`tests/v2/test_resume_clinic_router.py`, `test_resume_clinic_runner.py`, `test_resume_reviewer_agent.py`, `test_resume_clinic_repository.py`) are the **structural gate** — they assert wiring + invariants. This notebook is the **audit/inspection surface** where you read what the live model actually emits and decide whether the meaning is right (per ADR-058's pin/snapshot distinction).

## Setup

In [ ]:
import json
import os
from pprint import pprint

import httpx
import pandas as pd

BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
USER_ID = os.getenv("CLINIC_USER_ID", "0")  # the active profile under test

print(f"BASE_URL: {BASE_URL}")
print(f"USER_ID:  {USER_ID}")

# Sanity: the backend is up and responding.
_resp = httpx.get(f"{BASE_URL}/users", timeout=5.0)
_resp.raise_for_status()
_users = _resp.json().get("users") or []
print(f"Backend reachable. Profiles: {[(u['id'], u['name']) for u in _users]}")

## 1. Pick a resume

The clinic uses the active resume by default. If you want to target a specific one, set `RESUME_ID` below.

In [ ]:
# Leave as None to default to the user's active resume server-side.
RESUME_ID = None

# Optional: list this profile's resumes for reference.
import sqlite3
from pathlib import Path
DB = Path("data/v2.db")
if DB.exists():
    with sqlite3.connect(str(DB)) as conn:
        df = pd.read_sql_query(
            "SELECT id AS resume_id, file_name, version, COALESCE(is_active,0) AS active, created_at "
            "FROM resumes WHERE COALESCE(user_id,'0') = ? ORDER BY active DESC, created_at DESC",
            conn, params=(USER_ID,),
        )
    display(df)
else:
    print("data/v2.db not found — that's fine if the backend is on a different machine.")

## 2. Set targets (optional)

Leave `TARGET_ROLE` and `TARGET_TRACK` as `None` for quality-only mode (the alignment axis comes back as `null`). Fill them in to get the role/track alignment + the role-specific advice.

In [ ]:
TARGET_ROLE: str | None = "entry-level security analyst"
TARGET_TRACK: str | None = "ic"          # one of: ic, architect, management, or None
SENIORITY_AWARE: bool = True

## 3. Run the clinic

One POST. The backend runs `ResumeReviewerAgent` and then `FidelityReviewer` on the rewrites. Cost is attributed to this profile via a lightweight `workflow_runs` row (the runner generates one; you can see it in the Cost Dashboard afterward).

In [ ]:
payload = {"seniority_aware": SENIORITY_AWARE}
if RESUME_ID:
    payload["resume_id"] = RESUME_ID
if TARGET_ROLE:
    payload["target_role"] = TARGET_ROLE
if TARGET_TRACK:
    payload["target_track"] = TARGET_TRACK

with httpx.Client(timeout=180.0) as c:
    r = c.post(f"{BASE_URL}/users/{USER_ID}/resume-clinic", json=payload)
r.raise_for_status()
REVIEW = r.json()
CLINIC_ID = REVIEW["clinic_id"]
print(f"clinic_id:        {CLINIC_ID}")
print(f"workflow_run_id:  {REVIEW.get('workflow_run_id')}")
print(f"resume_id:        {REVIEW['resume_id']}")
print(f"target_role:      {REVIEW.get('target_role')}")
print(f"target_track:     {REVIEW.get('target_track')}")
print(f"seniority_aware:  {REVIEW.get('seniority_aware')}")

## 4. Quality scorecard — one row per dimension

The seven dimensions are `structure_ordering`, `impact_quantification`, `clarity`, `ats_formatting`, `consistency`, `length_fit`, `seniority_framing`. Each has a `rating` and lists of `findings` and `fixes`. Read the ratings first — anything `needs_work` deserves a fix.

In [ ]:
_quality = REVIEW.get("quality") or {}
print("OVERALL SUMMARY:\n")
print(_quality.get("overall_summary") or "(none)")
print()
_dims = _quality.get("dimensions") or []
if _dims:
    quality_df = pd.DataFrame([{
        "dimension": d.get("dimension"),
        "rating":    d.get("rating"),
        "#findings": len(d.get("findings") or []),
        "#fixes":    len(d.get("fixes") or []),
    } for d in _dims])
    display(quality_df)
    # Dump full findings + fixes per dimension.
    for d in _dims:
        print(f"\n— {d.get('dimension')} ({d.get('rating')}) —")
        for f in (d.get("findings") or []):
            print(f"  finding: {f}")
        for f in (d.get("fixes") or []):
            print(f"  fix:     {f}")

## 5. Role / track alignment (only when a target was given)

In [ ]:
_align = REVIEW.get("alignment")
if not _align:
    print("alignment is null — quality-only mode.")
else:
    print("fit_summary:\n", _align.get("fit_summary"))
    print("confidence:  ", _align.get("confidence"))
    for k in ("missing_skills", "missing_keywords", "emphasize",
             "suggested_certifications", "suggested_projects"):
        vs = _align.get(k) or []
        if vs:
            print(f"\n{k}:")
            for v in vs:
                print(f"  - {v}")

## 6. Reorganization plan

In [ ]:
_overhaul = REVIEW.get("overhaul") or {}
_reorg = _overhaul.get("reorganization") or {}
_order = _reorg.get("section_order") or []
if _order:
    print("Proposed section order:")
    for i, s in enumerate(_order, 1):
        print(f"  {i}. {s}")
_moves = _reorg.get("moves") or []
if _moves:
    print("\nMoves:")
    for m in _moves:
        print(f"  [{m.get('action')}] {m.get('subject')} — {m.get('rationale')}")

## 7. Rewrites — side-by-side

Every rewrite must carry `supporting_evidence` from the original resume. If you see an empty or generic-sounding evidence field, that is a semantic-drift smell — the schema requires `min_length=1`, but the model can satisfy that with `"see resume"`.

In [ ]:
_rewrites = _overhaul.get("rewrites") or []
if not _rewrites:
    print("No rewrites in this clinic run.")
else:
    rewrites_df = pd.DataFrame([{
        "section_label":       r.get("section_label"),
        "claim_type":          r.get("claim_type"),
        "original_text":       (r.get("original_text") or "")[:120],
        "suggested_text":      (r.get("suggested_text") or "")[:120],
        "supporting_evidence": (r.get("supporting_evidence") or "")[:120],
    } for r in _rewrites])
    display(rewrites_df)
    # Full text — easier to read for inspection.
    for i, r in enumerate(_rewrites, 1):
        print(f"\n--- rewrite {i} ({r.get('claim_type')}) — {r.get('section_label')} ---")
        print(f"ORIGINAL:  {r.get('original_text')}")
        print(f"SUGGESTED: {r.get('suggested_text')}")
        print(f"EVIDENCE:  {r.get('supporting_evidence')}")

## 8. Fidelity verdict

The reviewer polices the agent — not the human. If you submit an `edit` decision later, the human draft is trusted as final and is **not** re-checked.

Note: the Fidelity Reviewer prompt is tailoring-tuned, so some of its checks (length budget, impact rationale, strategy summary) apply less cleanly to clinic rewrites. The load-bearing fields here are `approval_recommendation`, `unsupported_claims`, and `fabricated_metrics`.

In [ ]:
_fid = REVIEW.get("fidelity_review")
if not _fid:
    print("No fidelity review on this row (no rewrites, or fidelity raised).")
else:
    print(f"recommendation: {_fid.get('approval_recommendation')}")
    print(f"confidence:     {_fid.get('confidence')}")
    for k in ("unsupported_claims", "fabricated_metrics", "inflated_scope_flags",
             "unsupported_technology_flags", "unsupported_certification_flags",
             "required_removals", "required_revisions"):
        vs = _fid.get(k) or []
        if vs:
            print(f"\n{k}:")
            for v in vs:
                print(f"  - {v}")

## 9. Record a decision (`approve`)

This persists `decision` and `decided_at` on the clinic row. `revise` and `reject` work the same way; `edit` requires an `edited` payload (next cell).

In [ ]:
with httpx.Client(timeout=10.0) as c:
    r = c.post(
        f"{BASE_URL}/resume-clinic/{CLINIC_ID}/decisions",
        json={"approval": "approve"},
    )
r.raise_for_status()
_approved = r.json()
print(f"decision:    {_approved.get('decision')}")
print(f"decided_at:  {_approved.get('decided_at')}")

## 10. (Optional) record an `edit` with a hand-crafted draft

The human-authored draft is stored alongside (not replacing) the agent's `overhaul`. ADR-059: this is NOT re-reviewed by Fidelity — the reviewer polices the agent, not the accountable human.

In [ ]:
edited_payload = {
    "reorganization": {
        "section_order": ["summary", "projects", "experience", "education", "skills"],
        "moves": [],
    },
    "rewrites": [
        # Whatever wording the human wants to lock in — this is the final draft.
    ],
}
with httpx.Client(timeout=10.0) as c:
    r = c.post(
        f"{BASE_URL}/resume-clinic/{CLINIC_ID}/decisions",
        json={"approval": "edit", "edited": edited_payload},
    )
r.raise_for_status()
_edited = r.json()
print(f"decision:   {_edited.get('decision')}")
print(f"edited:     {_edited.get('edited') is not None}")
print(f"overhaul retained for audit: {_edited.get('overhaul') is not None}")

## 11. List past clinic runs for this profile

In [ ]:
with httpx.Client(timeout=5.0) as c:
    r = c.get(f"{BASE_URL}/users/{USER_ID}/resume-clinic")
r.raise_for_status()
_past = r.json().get("reviews") or []
if _past:
    past_df = pd.DataFrame([{
        "clinic_id":     row["clinic_id"][:12],
        "target_role":   row.get("target_role") or "",
        "track":         row.get("target_track") or "",
        "decision":      row.get("decision") or "",
        "created_at":    row.get("created_at"),
    } for row in _past])
    display(past_df)
else:
    print("No past runs.")

## What to look for

When the model swaps (a new release, a config edit, anything that flips the pin in `tests/model_pins.json`), re-run this notebook against the new model and check:

- **Rating distribution** in the quality scorecard. If `needs_work` ratings collapse or explode without the resume changing, the model's interpretation of "adequate" has drifted. This is the canonical semantic-drift case the pin invariant was shipped to surface (per the Article 9 reviewer feedback).
- **Quantification rewrites** should always carry a placeholder when the source resume lacks a number (e.g. `"[N]+ services"`). A drifted model may start fabricating concrete numbers — Fidelity's `fabricated_metrics` should fire.
- **Alignment confidence** should track what the role-data block carries. With `NullRoleDataProvider` (v1) the reviewer relies on its own knowledge; the confidence should reflect that uncertainty.
- **`supporting_evidence`** longer than `"see resume"`. The schema's `min_length=1` is necessary but not sufficient — a drifted model can satisfy the constraint with one-token noise. Skim the evidence field on a handful of rewrites to confirm it points at concrete resume content.

If anything in the above feels off, the structural tests are not enough — write a new invariant test in `tests/v2/` so the next run catches it (per the cost-incident lesson).